# 01 · Are the examples aligned and independent?


Two excerpts from one upload may share a person, background and camera. Treating
them as independent test examples can reward recognition of the recording.
This stage establishes what one example is and keeps each source video in one
outer fold before any fitting. A source-video split does not establish a split
by verified person identity.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Detailed experiment specification](../../../notes/future-innovation-distillation/experiment-0-guide.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation/gate-v1")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, require_complete_report,
    )

## Execute this stage

Build eligible candidates, extract aligned poses, then freeze the cohort and source folds. This can take hours. The existing stages verify and reuse completed work; inspect the resulting exclusions and overlays below.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    run_stage("build-cohort", RUN_ROOT)
    run_stage("extract-poses", RUN_ROOT)

## 1. Count windows and sources separately

The real gate freezes 50 eligible windows and permits at most two from
a source, so at least 25 source videos are needed. Eligibility uses
alignment and observation quality, not teacher scores or condition labels.
This generated census uses the production fold assignment and cohort
validator. All excerpts from one source must receive the same fold.

In [ ]:
if MODE == "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_cohort import assign_source_folds, validate_cohort
    sources = [f"generated-source-{i // 2:02d}" for i in range(50)]
    fold_for = assign_source_folds(sources)
    cohort = pd.DataFrame({"window_id": [f"generated-window-{i}" for i in range(50)],
                           "video_id": sources, "outer_fold": [fold_for[s] for s in sources]})
    validate_cohort(cohort)
else:
    cohort = read_optional_table(RUN_ROOT, "manifests/gate-windows.csv")
if cohort is not None:
    census = cohort.groupby("outer_fold").agg(windows=("window_id", "size"),
                                              sources=("video_id", "nunique"))
    display(census)
    assert cohort.groupby("video_id").outer_fold.nunique().max() == 1
    ax = census.plot.bar(color=["#2f6f99", "#5f9e7e"], rot=0,
                         title="Generated outer held-out folds" if MODE == "teach" else "Saved outer held-out folds")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    ax.set(xlabel="Held-out fold", ylabel="Count")
    plt.tight_layout(); plt.show()
else:
    print("No frozen cohort manifest in this local copy. No census is inferred from images alone.")

## 2. Put the observation boundary on one timeline

Frames are zero-based after annotation conversion. Frames 0–31 supply
the inputs; the target spans 38–39 and ends eight frames after the last
observed frame. The teacher target uses the full 64-frame clip and can
therefore include information after frame 39. Its interpretation is a
full-clip contextual feature at that location.

The timeline reads the production `FRAME` contract, keeping the displayed
boundary tied to the implementation. The physical horizon in seconds
depends on the source frame rate.

In [ ]:
from gavd6_sjepa.research_directions.future_innovation.fi_contracts import FRAME
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.broken_barh([(0, FRAME.context_stop_exclusive)], (1, 0.65), facecolors="#2f6f99")
ax.broken_barh([(FRAME.target_tubelet_start, FRAME.tubelet_size)], (1, 0.65), facecolors="#e07a4b")
ax.broken_barh([(0, FRAME.frames_per_clip)], (0, 0.65), facecolors="#e6f2ea")
ax.set(yticks=[0.325, 1.325], yticklabels=["Teacher target context", "Past input / target location"],
       xticks=[0, 31, 38.5, 63], xticklabels=["0", "31", "38–39", "63"],
       xlabel="Zero-based frame", xlim=(0, 64))
ax.set_title("Inputs use the past; target encoding uses the full clip")
plt.tight_layout(); plt.show()

## 3. Inspect exclusions before interpreting model scores

A usable row needs exact decoding, retained person crops, aligned boxes
and adequate pose coverage. Natural missingness stays explicit through
confidence and validity channels. The pipeline saves every exclusion and
an alignment overlay for each accepted window.

Exclusion counts can reveal a narrow selected cohort. An overlay can
reveal a misplaced crop; it does not certify timing, source separation,
or teacher validity. Read the manifests as well as the pictures.

In [ ]:
if MODE != "teach":
    exclusions = read_optional_table(RUN_ROOT, "manifests/exclusions.csv")
    if exclusions is not None:
        display(exclusions.head(10))
        print(f"{len(exclusions)} recorded exclusions; showing at most ten.")
    overlays = sorted((RUN_ROOT / "qc/alignment-overlays").glob("*.jpg"))
    print(f"{len(overlays)} local overlays. Their presence alone does not establish a completed cohort.")
    if overlays:
        from IPython.display import Image
        display(Image(filename=str(overlays[0]), width=720))
    display(artifact_inventory(RUN_ROOT).iloc[:3])

## What this step establishes

In teaching mode the census is generated; in execution and inspection it describes the saved cohort. Examine exclusions and alignment alongside source counts. Missing manifests or inconsistent source assignments must be resolved before fitting. Next we test the teacher's observation boundary.

Continue with [02_teacher_features_and_validity.ipynb](02_teacher_features_and_validity.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")